### 주제

신용카드 사용자의 대금 연체 여부를 예측하는 AI 개발

### 데이터

- **train.csv [파일]**
    - ID : 신용카드를 보유한 고객의 고유 ID
    - TARGET : 고객의 신용카드 연체 여부
    - 성별
    - 차량 소유 여부
    - 부동산 소유 여부
    - 자녀 수
    - 연간 수입 : 단위 원
    - 수입 유형
    - 최종 학력
    - 결혼 여부
    - 주거 형태
    - 거주지 인구 비율: 고객이 거주하는 지역 인근의 인구 비율 (범위 0~1)
    - 휴대전화 소유 여부
    - 업무용 휴대폰 소유 여부
    - 이메일 소유 여부
    - 직업
    - 가족 구성원 수
    - 산업군 : 고객이 종사하는 직종의 산업군
    - 나이
    - 근속연수
    - 가입연수
    - 휴대전화 변경 경과일
    - 대출금액 : 단위 원
    - 대출연금 : 단위 원
    - 대출상품 가격(소비자 대출) : 단위 원
    - 종합 부동산 수치 : 부동산 관련 종합적인 수치를 정규화한 값 (범위 0~1)
    - 종합 신용평가 수치 : 신용평가 관련 종합적인 수치를 정규화한 값 (범위 0~1)
- **test.csv [파일]**
    - ID : 신용카드를 보유한 고객의 고유 ID
    - TARGET이 존재하지 않음
- **sample_submission.csv [파일] - 제출 양식**
    - ID : 신용카드를 보유한 고객의 고유 ID
    - TARGET : 고객의 신용카드 연체 여부를 예측하여 기입

### 코드 흐름

1. 환경 세팅
    - 폰트 설정
    - 시드 고정
2. 데이터 불러오기
    - train, test, submission 데이터 불러오기
3. ID 칼럼 삭제
    - ID는 예측에 의미 없음
4. 범주형 인코딩
    - 이진 범주형 인코딩
        - 성별 값을 0, 1 같은 숫자로 변환

In [ ]:
binary_col = ['성별']
for column in binary_col:
	unique_values = train[column].unique()
	value_mapping = {value: idx for idx, value in enumerate(unique_values)}

	train[column] = train[column].replace(value_mapping)
	test[column] = test[column].replace(value_mapping)

  - 순서형 범주 인코딩
      - 최종 학력에 순서를 부여해서 숫자로 변환

In [ ]:
edu_order = ['저학력자', '고등학교 졸업', '대학교 중퇴', '대학교 졸업 이상']
edu_mapping = {edu: idx for idx, edu in enumerate(edu_order)}
train['최종 학력'] = train['최종 학력'].replace(edu_mapping)
test['최종 학력'] = test['최종 학력'].replace(edu_mapping)

  - 나머지 범주형 변수 Binary Encoding
      - object 타입 범주형 변수들을 BinaryEncoder로 변환

In [ ]:
# 바이너리 인코딩 수행
import category_encoders as ce

# 선택할 범주형 칼럼들
categorical_cols = train.select_dtypes(include='object').columns.tolist()

encoder = ce.BinaryEncoder(cols=categorical_cols)

binary_train = encoder.fit_transform(train[categorical_cols])
binary_test = encoder.transform(test[categorical_cols])

train = pd.concat([train, binary_train], axis=1)
test = pd.concat([test, binary_test], axis=1)

categorical_cols.extend(['휴대전화 소유 여부'])
train.drop(categorical_cols, axis=1, inplace=True)
test.drop(categorical_cols, axis=1, inplace=True)

5. 피처 엔지니어링
    - 가족 구성원 수 상한 처리
    - befor_employed 파생변수 생성
    - KMeans 클러스터링
6. Box-Cox 변환 + 스케일링
    - 연간 수입 Box-Cox 변환
    - 입력/타깃 분리
    - StandardScaler 적용
7. LGBM 모델 학습
8. 테스트셋 예측
9. 제출 파일 저장
### 새롭게 알게 된 내용 / 어려운 내용 / 배울 점

- 범주형 처리 흐름이 명확해서 좋았고, 순서형 변수와 일반 범주형 변수를 다르게 처리한 점이 좋았다.
- Box-Cox처럼 분포를 보정한 것이 인상적이었다.
- 다만, 성능 검증 과정이 빠져있어 성능 검증을 했다면 더 좋았을 것 같다.